# Team 03 - Data Understanding & Model Team Liaison Notebook

**Team:** Data Understanding / Model Liaison  
**Worker name:** _fill in_  
**Date:** _fill in_  

This team's job:
1. Evaluate the final corpus composition across all 4 datasets
2. Confirm mixing ratios with the Model team
3. Validate tokenizer compatibility
4. Produce the handoff data card for the Model team


## Step 1 - Corpus Stats Across All Datasets

In [ ]:
import gzip, json
from pathlib import Path
from collections import defaultdict

SHARDS_ROOT = Path("/mnt/ssd/shards")  # output of Team 02

SOURCES = ["thaillm", "seapile", "thestack_v2", "finemath"]

stats = {}
for source in SOURCES:
    source_dir = SHARDS_ROOT / source
    if not source_dir.exists():
        print(f"[SKIP] {source} - not found at {source_dir}")
        continue
    
    n_docs = n_chars = 0
    for shard in source_dir.glob("*.jsonl.gz"):
        if ".spam." in shard.name:
            continue
        with gzip.open(shard, "rt") as f:
            for line in f:
                doc = json.loads(line)
                n_docs  += 1
                n_chars += doc.get("n_chars", 0)
    
    # Rough token estimate: 1 token ~ 4 chars for Thai/English mix
    n_tokens_est = n_chars / 4
    stats[source] = {"n_docs": n_docs, "n_chars": n_chars, "n_tokens_est": n_tokens_est}

total_tokens = sum(v["n_tokens_est"] for v in stats.values())

print(f"{'Source':20s} {'Docs':>10s} {'Chars':>14s} {'Tokens (est)':>14s} {'Mix %':>8s}")
print("-" * 70)
for src, s in stats.items():
    mix_pct = s["n_tokens_est"] / max(total_tokens, 1) * 100
    print(f"{src:20s} {s['n_docs']:>10,} {s['n_chars']:>14,} {s['n_tokens_est']:>14,.0f} {mix_pct:>7.1f}%")
print("-" * 70)
print(f"{'TOTAL':20s} {sum(v['n_docs'] for v in stats.values()):>10,} {' ':>14s} {total_tokens:>14,.0f} {'100.0%':>8s}")

## Step 2 - Compare Actual Mix vs Target Mix

Target: THAILLM 35%, SEAPILE 25%, Stack V2 25%, FineMath 15%

In [ ]:
TARGET_MIX = {
    "thaillm":     0.35,
    "seapile":     0.25,
    "thestack_v2": 0.25,
    "finemath":    0.15,
}

print(f"{'Source':20s} {'Target':>8s} {'Actual':>8s} {'Delta':>8s} {'Status':>8s}")
print("-" * 60)
for src in SOURCES:
    if src not in stats:
        continue
    actual = stats[src]["n_tokens_est"] / max(total_tokens, 1)
    target = TARGET_MIX.get(src, 0)
    delta  = actual - target
    status = "OK" if abs(delta) < 0.05 else "REVIEW"
    print(f"{src:20s} {target:>7.1%} {actual:>7.1%} {delta:>+7.1%} {status:>8s}")

print()
print("Note: delta > 5pp = discuss with Calvin and Model team before handoff")

## Step 3 - Thai Character Ratio Check (SEAPILE + THAILLM)

In [ ]:
import re
import random

THAI_RE = re.compile(r"[\u0E00-\u0E7F]")

for source in ["thaillm", "seapile"]:
    source_dir = SHARDS_ROOT / source
    if not source_dir.exists():
        continue
    
    shards = [s for s in source_dir.glob("*.jsonl.gz") if ".spam." not in s.name]
    if not shards:
        continue
    
    # Sample 200 docs from first shard
    with gzip.open(shards[0], "rt") as f:
        sample_docs = [json.loads(l) for l in f][:200]
    
    ratios = []
    for doc in sample_docs:
        text = doc.get("text", "")
        thai_n = len(THAI_RE.findall(text))
        ratios.append(thai_n / max(len(text), 1))
    
    avg = sum(ratios) / max(len(ratios), 1)
    low = min(ratios)
    high = max(ratios)
    print(f"{source:15s} Thai char ratio - avg: {avg:.2f}  min: {low:.2f}  max: {high:.2f}")

print()
print("Expected: THAILLM avg > 0.5, SEAPILE avg > 0.3 (multilingual)")

## Step 4 - Confirm Format Compatibility with Model Team

Run this and share the output directly with the Model team (Boss, New, Iccue) to confirm they can load the shards.

In [ ]:
# Test loading one shard the way the Model team will
from torch.utils.data import Dataset  # optional - remove if torch not installed

# Simple load test
test_shard = next((SHARDS_ROOT / "thaillm").glob("*.jsonl.gz"), None)
if test_shard:
    with gzip.open(test_shard, "rt") as f:
        docs = [json.loads(l) for l in f]
    
    print(f"Loaded {len(docs):,} docs from {test_shard.name}")
    print(f"Schema keys  : {list(docs[0].keys())}")
    print(f"Sample text  : {docs[0]['text'][:120]}")
    print(f"n_chars range: {min(d['n_chars'] for d in docs)} - {max(d['n_chars'] for d in docs)}")
    print()
    print("-- Confirm with Model team that above schema is acceptable --")

## Step 5 - Produce Handoff Data Card

In [ ]:
from datetime import datetime

card = {
    "handoff_time": datetime.now().isoformat(),
    "prepared_by":  "",  # your name
    "confirmed_by_model_team": False,  # set True after verbal confirm
    "corpus": [
        {
            "source":       src,
            "n_docs":       stats[src]["n_docs"],
            "n_tokens_est": int(stats[src]["n_tokens_est"]),
            "mix_target":   TARGET_MIX.get(src, 0),
            "mix_actual":   round(stats[src]["n_tokens_est"] / max(total_tokens, 1), 4),
        }
        for src in SOURCES if src in stats
    ],
    "total_tokens_est": int(total_tokens),
    "output_format":   "JSONL.gz, UTF-8, NFC, one doc per line",
    "schema_fields":   ["id","source","license","text","n_chars","n_words","quality_flags","dup_group","domain"],
    "tokenizer_note":  "SentencePiece UNIGRAM, vocab=48k, byte_fallback=True. Train on raw text - do NOT pre-segment Thai.",
    "shard_path":      str(SHARDS_ROOT),
    "sha256sums_path": str(SHARDS_ROOT / "SHA256SUMS"),
}

card_path = Path("../../logs/handoff_data_card.json")
card_path.parent.mkdir(parents=True, exist_ok=True)
card_path.write_text(json.dumps(card, indent=2, ensure_ascii=False))

print(json.dumps(card, indent=2, ensure_ascii=False))
print(f"\nSaved to {card_path}")